
# Final Retrieval Evaluation

**Origin - Independent Test-Set Evaluation**

This notebook evaluates Origin's frozen language-aware retrieval pipeline on the reserved final test set.

### Retrieval Strategy

- **Arabic:** Chunked Semantic Search.
- **English:** BM25 + Chunked Semantic Search using Reciprocal Rank Fusion (RRF).

The approved implementation is loaded from `src/retrieval.py`, using the configuration selected during development.

### Evaluation Approach

The system is evaluated on 20 reserved test queries (10 Arabic and 10 English). Reference Hit@5 and MRR@5 are calculated for queries with human-selected reference project IDs.

The retrieval configuration remains unchanged throughout final testing. RAG-generated responses will be evaluated separately.


## 1. Setup and dependencies



In [1]:
%pip install -q sentence-transformers rank-bm25

In [2]:
import pandas as pd
import numpy as np
import re
import torch

from pathlib import Path
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

print("Libraries imported successfully!")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


Libraries imported successfully!
Device: cpu



## 2. Load and Validate Datasets

Load the historical graduation projects and evaluation queries from GitHub. Verify dataset integrity and ensure that the reserved final test queries are separate from the development queries.


In [3]:
# Load datasets from GitHub
base_url = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
)

projects_df = pd.read_csv(
    base_url + "data/processed/projects_clean.csv",
    encoding="utf-8-sig"
)

eval_df = pd.read_csv(
    base_url + "data/evaluation/evaluation_queries.csv",
    encoding="utf-8-sig"
)

# Standardize column names
projects_df.columns = projects_df.columns.str.strip().str.lower()
eval_df.columns = eval_df.columns.str.strip().str.lower()

# Validate project dataset
assert projects_df.shape == (103, 7)
assert projects_df["project_id"].is_unique
assert projects_df[["title", "abstract"]].notna().all().all()

# Validate evaluation dataset
assert len(eval_df) == 60
assert eval_df["query_id"].is_unique

# Separate development and final test queries
dev_queries = eval_df[
    eval_df["split"] == "dev"
].copy().reset_index(drop=True)

final_test_queries = eval_df[
    eval_df["split"] == "final_test"
].copy().reset_index(drop=True)

assert len(dev_queries) == 40
assert len(final_test_queries) == 20

# Verify split separation
assert set(dev_queries["query_id"]).isdisjoint(
    set(final_test_queries["query_id"])
)

# Validate final test inputs
assert final_test_queries[
    ["query_title", "query_description", "language"]
].notna().all().all()

assert final_test_queries["language"].isin(
    ["ar", "en"]
).all()

print("Datasets loaded and validated successfully!")
print("Historical projects:", len(projects_df))
print("Development queries:", len(dev_queries))
print("Final test queries:", len(final_test_queries))
print("Arabic final test queries:",
      (final_test_queries["language"] == "ar").sum())
print("English final test queries:",
      (final_test_queries["language"] == "en").sum())
print("Development and final test splits are separate.")


Datasets loaded and validated successfully!
Historical projects: 103
Development queries: 40
Final test queries: 20
Arabic final test queries: 10
English final test queries: 10
Development and final test splits are separate.



## 3. Load Frozen Retrieval Pipeline

Load Origin's approved retrieval implementation from `src/retrieval.py` at the pinned GitHub commit.

The same language-aware configuration selected during development is reused without modifying preprocessing, model parameters, or ranking methods.



### 3.1. Import Frozen Retrieval Module


In [4]:
import sys
import subprocess
from pathlib import Path

# Retrieve the frozen implementation
repo_dir = Path("/content/Origin")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "-q",
         "https://github.com/esrrahafase-crypto/Origin.git",
         str(repo_dir)],
        check=True
    )

# Pin the approved retrieval implementation
subprocess.run(
    ["git", "-C", str(repo_dir), "checkout", "-q",
     "b69b27686fa20c235b72174077bfc00c84a1c00f"],
    check=True
)

# Import the reusable retrieval module
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

from src.retrieval import build_retrieval_pipeline

assert callable(build_retrieval_pipeline)

print("Frozen retrieval module imported successfully!")
print("Source: src/retrieval.py")
print("Retrieval implementation is ready for validation.")


Frozen retrieval module imported successfully!
Source: src/retrieval.py
Retrieval implementation is ready for validation.



### 3.2. Initialize Frozen Retrieval Pipeline


In [5]:

import gc

# Release the previously loaded model to avoid duplicate GPU usage
if "model" in globals():
    del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Initialize the frozen retrieval pipeline
retrieval = build_retrieval_pipeline(projects_df)

# Validate model and embeddings
assert retrieval.model_name == (
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)
assert retrieval.model.max_seq_length == 128
assert retrieval.chunk_overlap == 20
assert retrieval.chunked_project_embeddings.shape == (103, 768)

# Validate retrieval functions
assert callable(retrieval.semantic_search)
assert callable(retrieval.bm25_search)
assert callable(retrieval.hybrid_search)
assert callable(retrieval.language_aware_search)

# Validate project alignment
assert retrieval.hybrid_df["project_id"].tolist() == (
    projects_df["project_id"].tolist()
)

print("Frozen retrieval pipeline initialized successfully!")
print("Projects:", len(retrieval.hybrid_df))
print("Embeddings shape:", retrieval.chunked_project_embeddings.shape)
print("Chunk overlap:", retrieval.chunk_overlap)
print("All retrieval functions are available.")


BM25 PREPROCESSING COMPLETED
Total projects: 103
Tokenized documents: 103
Sample tokens: ['automatic', 'plant', 'watering', 'tracking', 'people', 'struggle', 'watering', 'plants', 'regularly', 'usually', 'leads', 'plants', 'unhealthy', 'dying', 'address', 'common', 'issue', 'important', 'monitor', 'basic']
BM25 index created successfully!
Indexed projects: 103
English BM25 retrieval is ready!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/content/Origin/src/retrieval.py:149: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (208 > 128). Running this sequence through the model will result in indexing errors


Semantic model loaded successfully!
Device: cpu
Embedding dimension: 768
Project chunking completed successfully!
Total projects: 103
Total chunks: 298
Average chunks per project: 2.89
Maximum tokens: 128


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Chunked project embeddings generated successfully!
Embeddings shape: (103, 768)
Chunked semantic retrieval is ready!
Hybrid search updated successfully!
Language-aware retrieval is ready!
Frozen retrieval pipeline initialized successfully!
Projects: 103
Embeddings shape: (103, 768)
Chunk overlap: 20
All retrieval functions are available.



## 4. Development Reproducibility Check


In [6]:
# Verify frozen retrieval performance on development queries
evaluation_records = []

for _, query in dev_queries.iterrows():
    results = retrieval.language_aware_search(
        title=query["query_title"],
        description=query["query_description"],
        language=query["language"],
        top_k=5
    )

    refs = query["reference_project_ids"]

    if pd.isna(refs) or not str(refs).strip():
        continue

    reference_ids = {
        ref.strip()
        for ref in str(refs).split(";")
        if ref.strip()
    }

    matched_ranks = results.loc[
        results["project_id"].isin(reference_ids),
        "rank"
    ]

    first_rank = (
        int(matched_ranks.min())
        if not matched_ranks.empty else None
    )

    evaluation_records.append({
        "query_id": query["query_id"],
        "language": query["language"],
        "reference_hit_at_5": int(first_rank is not None),
        "reference_mrr_at_5": (
            1.0 / first_rank if first_rank is not None else 0.0
        )
    })

reproducibility_eval = pd.DataFrame(evaluation_records)

reproducibility_summary = (
    reproducibility_eval
    .groupby("language")
    .agg(
        evaluated_queries=("query_id", "count"),
        reference_hit_at_5=("reference_hit_at_5", "mean"),
        reference_mrr_at_5=("reference_mrr_at_5", "mean")
    )
    .reset_index()
)

# Verify expected development metrics
expected = {
    "ar": (0.9231, 0.6141),
    "en": (0.9231, 0.8462)
}

assert len(reproducibility_eval) == 26

for _, row in reproducibility_summary.iterrows():
    hit, mrr = expected[row["language"]]

    assert row["evaluated_queries"] == 13
    assert np.isclose(row["reference_hit_at_5"], hit, atol=0.0001)
    assert np.isclose(row["reference_mrr_at_5"], mrr, atol=0.0001)

print("Frozen retrieval reproducibility check passed!")
display(reproducibility_summary.round(4))


Frozen retrieval reproducibility check passed!


,language,evaluated_queries,reference_hit_at_5,reference_mrr_at_5
0,ar,13,0.9231,0.6141
1,en,13,0.9231,0.8462



## 5. Final Test Retrieval

Evaluate the frozen language-aware retrieval pipeline on the 20 reserved final-test queries.

The retrieval configuration remains unchanged, and the top five historical projects are retrieved for each query.


In [8]:
# Run frozen retrieval on final-test queries
final_test_records = []

for _, query in final_test_queries.iterrows():
    results = retrieval.language_aware_search(
        title=query["query_title"],
        description=query["query_description"],
        language=query["language"],
        top_k=5
    ).copy()

    results.insert(0, "query_id", query["query_id"])
    results.insert(1, "language", query["language"])

    final_test_records.append(results)

final_test_results = pd.concat(
    final_test_records,
    ignore_index=True
)

# Validate retrieval coverage
assert len(final_test_results) == 100
assert final_test_results["query_id"].nunique() == 20
assert final_test_results.groupby("query_id").size().eq(5).all()
assert not final_test_results.duplicated(
    ["query_id", "project_id"]
).any()

print("Final-test retrieval completed successfully!")
print("Final-test queries:", len(final_test_queries))
print("Total retrieved projects:", len(final_test_results))
print("Arabic queries:",
      (final_test_queries["language"] == "ar").sum())
print("English queries:",
      (final_test_queries["language"] == "en").sum())

display(
    final_test_results[
        [
            "query_id",
            "language",
            "rank",
            "project_id",
            "title",
            "retrieval_mode"
        ]
    ].head(20)
)


Final-test retrieval completed successfully!
Final-test queries: 20
Total retrieved projects: 100
Arabic queries: 10
English queries: 10


,query_id,language,rank,project_id,title,retrieval_mode
0,Q001,en,1,P001,Automatic Plant Watering and Tracking System,hybrid_en
1,Q001,en,2,P058,نَبْت A Machine Learning-Based Plant Recommend...,hybrid_en
2,Q001,en,3,P089,AI-Powered Platform for Plant Identification a...,hybrid_en
3,Q001,en,4,P008,SyncUp,hybrid_en
4,Q001,en,5,P085,Driver drowsiness detection system,hybrid_en
5,Q002,en,1,P029,Collab: A Revolutionary web-based platform des...,hybrid_en
6,Q002,en,2,P008,SyncUp,hybrid_en
7,Q002,en,3,P086,عَون AWN Volunteer management website,hybrid_en
8,Q002,en,4,P072,AutoReview,hybrid_en
9,Q002,en,5,P041,PNU ChatBot,hybrid_en



## 6. Final Test Evaluation

Evaluate the frozen language-aware retrieval pipeline using human-selected reference project IDs from the reserved final test set.

Calculate Reference Hit@5 and Reference MRR@5 separately for Arabic and English queries. Report the number of evaluated queries to distinguish reference coverage from retrieval performance.


In [9]:
# Evaluate final-test retrieval against human-selected references
evaluation_records = []

for _, query in final_test_queries.iterrows():
    refs = query["reference_project_ids"]

    # Skip queries without reference annotations
    if pd.isna(refs) or not str(refs).strip():
        continue

    reference_ids = {
        ref.strip()
        for ref in str(refs).split(";")
        if ref.strip()
    }

    query_results = final_test_results[
        final_test_results["query_id"] == query["query_id"]
    ].sort_values("rank")

    matched_ranks = query_results.loc[
        query_results["project_id"].isin(reference_ids),
        "rank"
    ]

    first_rank = (
        int(matched_ranks.min())
        if not matched_ranks.empty else None
    )

    evaluation_records.append({
        "query_id": query["query_id"],
        "language": query["language"],
        "reference_hit_at_5": int(first_rank is not None),
        "reference_mrr_at_5": (
            1.0 / first_rank if first_rank is not None else 0.0
        )
    })

final_test_evaluation = pd.DataFrame(evaluation_records)

# Summarize evaluation by language
final_test_summary = (
    final_test_evaluation
    .groupby("language")
    .agg(
        evaluated_queries=("query_id", "count"),
        reference_hit_at_5=("reference_hit_at_5", "mean"),
        reference_mrr_at_5=("reference_mrr_at_5", "mean")
    )
    .reset_index()
)

# Validate evaluation coverage
assert len(final_test_evaluation) == 14
assert set(final_test_summary["language"]) == {"ar", "en"}
assert final_test_summary["evaluated_queries"].eq(7).all()

print("Final-test reference evaluation completed successfully!")
print("Total final-test queries:", len(final_test_queries))
print("Queries with reference annotations:", len(final_test_evaluation))
print("Queries without reference annotations:",
      len(final_test_queries) - len(final_test_evaluation))

display(final_test_summary.round(4))


Final-test reference evaluation completed successfully!
Total final-test queries: 20
Queries with reference annotations: 14
Queries without reference annotations: 6


,language,evaluated_queries,reference_hit_at_5,reference_mrr_at_5
0,ar,7,0.7143,0.6190
1,en,7,1.0000,0.9048



## 7. Final Test Validation and Error Analysis

Validate the final-test retrieval outputs and identify queries where no human-selected reference project appeared within the top five results.

This analysis is descriptive only and will not be used to tune the frozen retrieval configuration.


In [10]:
# Validate final-test results
assert len(final_test_results) == 100
assert final_test_results["query_id"].nunique() == 20
assert final_test_results.groupby("query_id").size().eq(5).all()

# Identify queries with missed references
missed_test_queries = final_test_evaluation[
    final_test_evaluation["reference_hit_at_5"] == 0
].copy()

missed_test_details = missed_test_queries.merge(
    final_test_queries[
        ["query_id", "query_title", "reference_project_ids"]
    ],
    on="query_id",
    how="left"
)

print("Final-test validation passed!")
print("Queries evaluated with references:", len(final_test_evaluation))
print("Queries with missed references:", len(missed_test_details))

display(
    missed_test_details[
        [
            "query_id",
            "language",
            "query_title",
            "reference_project_ids",
            "reference_hit_at_5",
            "reference_mrr_at_5"
        ]
    ]
)


Final-test validation passed!
Queries evaluated with references: 14
Queries with missed references: 2


,query_id,language,query_title,reference_project_ids,reference_hit_at_5,reference_mrr_at_5
0,Q017,ar,دليل تهيئة القاعات للصم,P047;P074,0,0.0
1,Q048,ar,منظم استعارة أدوات الأندية,P048,0,0.0



## 8. Save Final Evaluation Results

Export the frozen retrieval outputs, reference-based evaluation metrics, and missed-reference cases from the reserved final test set.

These results are preserved without modifying or retuning the selected retrieval pipeline.


In [11]:
from pathlib import Path
import json

# Create output directory
output_dir = Path("/content/origin_final_evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

# Save final-test retrieval results
final_test_results.to_csv(
    output_dir / "final_test_top5.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save query-level reference evaluation
final_test_evaluation.to_csv(
    output_dir / "final_test_query_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save language-level evaluation summary
final_test_summary.to_csv(
    output_dir / "final_test_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save missed-reference cases
missed_test_details.to_csv(
    output_dir / "final_test_missed_references.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save evaluation metadata
evaluation_metadata = {
    "retrieval_strategy": "language_aware",
    "selection_split": "dev",
    "evaluation_split": "final_test",
    "total_test_queries": 20,
    "annotated_queries": 14,
    "unannotated_queries": 6,
    "top_k": 5,
    "metrics": ["reference_hit_at_5", "reference_mrr_at_5"],
    "configuration_tuned_on_final_test": False
}

with open(
    output_dir / "evaluation_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(evaluation_metadata, f, indent=4)

# Validate saved files
assert len(list(output_dir.iterdir())) == 5

print("Final evaluation results saved successfully!")

for file in sorted(output_dir.iterdir()):
    print(file.name)


Final evaluation results saved successfully!
evaluation_metadata.json
final_test_missed_references.csv
final_test_query_metrics.csv
final_test_summary.csv
final_test_top5.csv



## 9. Conclusions

This notebook evaluated Origin's frozen language-aware retrieval pipeline on the reserved final test set, following successful reproduction of the development results.

### Evaluation Setup

- The approved retrieval implementation was loaded from the pinned `src/retrieval.py` version.
- **Arabic queries:** Chunked Semantic Search.
- **English queries:** BM25 + Chunked Semantic Search combined using RRF (`k=60`).
- The final test set contained 20 queries (10 Arabic and 10 English).
- Reference Hit@5 and MRR@5 were calculated using 14 queries with human-selected reference project IDs (7 per language).

### Final Test Results

| Metric | Arabic | English |
|---|---:|---:|
| Total Test Queries | 10 | 10 |
| Queries with References | 7 | 7 |
| Reference Hit@5 | 0.7143 | 1.0000 |
| Reference MRR@5 | 0.6190 | 0.9048 |

### Key Findings

- The pipeline successfully retrieved five historical projects for each of the 20 final-test queries.
- English retrieval achieved a Reference Hit@5 of 1.0000, identifying at least one selected reference project for all seven annotated queries.
- Arabic retrieval achieved a Reference Hit@5 of 0.7143, identifying a reference project for five of seven annotated queries.
- Two Arabic queries (Q017 and Q048) missed their selected references within the top five results.
- The retrieval configuration remained unchanged throughout final testing.

### Limitations and Next Steps

- Only 14 of 20 test queries had reference annotations; the remaining six were excluded from reference-based metrics.
- Human-selected references may not cover every relevant historical project, and the small evaluation sample limits generalization.
- The frozen retrieval pipeline will be integrated into Origin's RAG system to support evidence-grounded recommendations for improving graduation project ideas.
- Generated responses will be evaluated separately for relevance, faithfulness, and practical usefulness.

Overall, the final test demonstrated strong reference-based English retrieval and more variable Arabic performance. The selected language-aware configuration was retained without test-driven adjustments and will serve as the retrieval foundation for Origin's RAG component.
